In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install uv

In [ ]:
# Install Telegram API client
!uv pip install telethon

# Install fuzzy matching for OCR / name errors
!uv pip install fuzzywuzzy python-Levenshtein


In [ ]:
# For environment variables (security)
import os

# For regular expressions (hashtags, cleaning text)
import re

# Telegram client library
from telethon import TelegramClient

# For fuzzy text matching
from fuzzywuzzy import fuzz

# For file and folder operations
import shutil
from pathlib import Path


In [ ]:
# Store API ID in environment
os.environ["TG_API_ID"] = "PUT_API_ID_HERE"

# Store API HASH in environment
os.environ["TG_API_HASH"] = "PUT_API_HASH_HERE"


In [ ]:
# Read API ID from environment variables
api_id = int(os.getenv("TG_API_ID"))

# Read API HASH from environment variables
api_hash = os.getenv("TG_API_HASH")

# Stop execution if credentials are missing
assert api_id and api_hash, "Telegram API credentials not found"


In [ ]:
# Create Telegram client session
client = TelegramClient("session", api_id, api_hash)

# Start client (asks for login code first time only)
await client.start()

# Confirmation message
print("Connected to Telegram successfully ✅")


In [ ]:
# Function to normalize Arabic/English text for matching
def normalize(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and symbols
    text = re.sub(r"[^\w\s]", " ", text)

    # Replace multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    return text.strip()


In [ ]:
# Telegram channel username or ID
channel = "Legal_Knowledgee"

# Dictionary: normalized book title -> categories
book_map = {}

# Iterate over channel messages
async for message in client.iter_messages(channel, limit=5000):

    # Skip empty messages
    if not message.text:
        continue

    # Extract hashtags
    hashtags = re.findall(r"#\S+", message.text)

    # Skip messages without hashtags
    if not hashtags:
        continue

    # 🚫 Skip index / guide messages (too many hashtags)
    if len(hashtags) > 5:
        continue

    # First line is assumed to be the book title
    title = message.text.split("\n")[0].strip()

    # Normalize title
    title = normalize(title)

    # Remove # from hashtag names
    categories = [tag.replace("#", "") for tag in hashtags]

    # Save mapping
    book_map[title] = categories

print(f"Indexed {len(book_map)} real book messages ✅")


In [ ]:
input_folder = Path("/content/drive/MyDrive/Graduation Project/Collected Data")
output_folder = Path("/content/drive/MyDrive/Graduation Project/Categories")


In [ ]:
# Find best matching Telegram title for a file name
def find_best_match(file_name):
    best_score = 0
    best_match = None

    # Compare file name with every Telegram title
    for title in book_map:
        score = fuzz.partial_ratio(file_name, title)

        # Keep highest similarity score
        if score > best_score:
            best_score = score
            best_match = title

    # Only accept strong matches
    return best_match if best_score >= 75 else None


In [ ]:
moved = 0
skipped = 0

for file in input_folder.glob("*.txt"):

    # Skip if file already moved in a previous run
    if not file.exists():
        continue

    # Normalize file name
    normalized_name = normalize(file.stem)

    # Find matching Telegram title
    match = find_best_match(normalized_name)

    # If no match → skip
    if not match:
        skipped += 1
        continue

    # ✅ TAKE FIRST CATEGORY ONLY
    category = book_map[match][0]

    # Create category folder (IMPORTANT: parents=True)
    category_folder = output_folder / category
    category_folder.mkdir(parents=True, exist_ok=True)

    # Destination path
    destination = category_folder / file.name

    # Move file ONCE
    shutil.move(str(file), str(destination))

    moved += 1

print(f"Moved: {moved} files")
print(f"Skipped: {skipped} files")


In [ ]:
pdf_files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]
print(f"Found {len(pdf_files)} PDF files to process\n")

# Second try

In [ ]:
import unicodedata

# Async support
import asyncio
def normalize(text):
    """
    Normalize Arabic text:
    - remove tashkeel
    - unify Arabic letters
    - remove symbols
    - lowercase
    """
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


In [ ]:
channel = "Legal_Knowledgee"

book_map = {}

async def build_book_map():
    async for message in client.iter_messages(channel, limit=5000):

        if not message.text:
            continue

        text = message.text

        # Extract hashtags with positions
        hashtags = [(m.group(), m.start()) for m in re.finditer(r"#\S+", text)]

        if not hashtags:
            continue

        # Extract possible file titles (lines without hashtags)
        lines = text.split("\n")

        for i, line in enumerate(lines):
            if "#" in line:
                continue

            title = normalize(line)
            if len(title) < 5:
                continue

            # Find nearest hashtag AFTER this line
            line_pos = text.find(line)
            nearest = None

            for tag, pos in hashtags:
                if pos > line_pos:
                    nearest = tag.replace("#", "")
                    break

            if nearest:
                book_map[title] = nearest

    print(f"Mapped {len(book_map)} files from Telegram")

await build_book_map()


In [ ]:
moved = 0
skipped = 0

for file in input_folder.glob("*.txt"):

    # Normalize file name
    file_name = normalize(file.stem)

    # Try to find category
    category = book_map.get(file_name)

    if not category:
        skipped += 1
        continue

    # Category folder path
    category_folder = output_folder / category

    # Create folder ONLY if missing
    category_folder.mkdir(exist_ok=True)

    destination = category_folder / file.name

    # Skip if already moved
    if destination.exists():
        skipped += 1
        continue

    # MOVE file (not copy)
    shutil.move(str(file), str(destination))

    moved += 1

print(f"Moved: {moved}")
print(f"Skipped: {skipped}")
